In [ ]:
# !pip install nibabel opencv-python tqdm


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os 
import numpy as np
import nibabel as nib
import cv2
from tqdm import tqdm

In [ ]:
DATA_PATH = "../Liver_Dataset"
IMAGE_PATH = os.path.join(DATA_PATH,"imagesTr")
MASK_PATH = os.path.join(DATA_PATH,"labelsTr")

OUTPUT_PATH = "../processed_data"
os.makedirs(OUTPUT_PATH,exist_ok = True)

In [3]:
# Patient-wise split
patients = sorted(os.listdir(IMAGE_PATH))

np.random.seed(42)
np.random.shuffle(patients)

train_split = int(0.7 * len(patients))
val_split = int(0.85*len(patients))

train_patients = patients[:train_split]
val_patients = patients[train_split:val_split]
test_patients = patients[val_split:]

In [4]:
# Folder structure create
def create_dirs(split):
    os.makedirs(f"{OUTPUT_PATH}/{split}/images",exist_ok = True)
    os.makedirs(f"{OUTPUT_PATH}/{split}/masks",exist_ok = True)

for split in ["train","val","test"]:
    create_dirs(split)

In [5]:
# function to convert 3D volumes to 2D slices 
import albumentations as A

# Augmentation pipeline
augment = A.Compose([
    A.HorizontalFlip(p = 0.5),
    A.VerticalFlip(p = 0.5),
    A.RandomRotate90(p = 0.5),
    A.ElasticTransform(p = 0.3),
    A.RandomBrightnessContrast(p = 0.3),
    A.GaussNoise(p = 0.3),
])
def process_patient(patient_id,split):
    img_path = os.path.join(IMAGE_PATH, patient_id)
    mask_path = os.path.join(MASK_PATH, patient_id)

    image_3d = nib.load(img_path).get_fdata()
    mask_3d = nib.load(mask_path).get_fdata()

    depth = image_3d.shape[2]

    for i in range(depth):
        img_slice = image_3d[:,:,i]
        mask_slice = mask_3d[:,:,i]

        # skip empty slices
        if mask_slice.sum() == 0:
            continue

        #normalize
        img_slice = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-8)

        #resize
        img_slice = cv2.resize(img_slice, (256,256))
        mask_slice = cv2.resize(mask_slice , (256,256), interpolation = cv2.INTER_NEAREST)

        #convert to unit8
        img_slice = (img_slice*255).astype(np.uint8)
        mask_slice = mask_slice.astype(np.uint8)
        
        #save
        filename = f"{patient_id[:-7]}_{i}"

        #class Balance Logic
        tumor_pixels = np.sum(mask_slice == 2)
        
        if tumor_pixels > 100:
            repeat = 2 
        else:
            repeat = 1 
        
        for r in range(repeat):
            cv2.imwrite(f"{OUTPUT_PATH}/{split}/images/{filename}_{r}.png", img_slice)
            cv2.imwrite(f"{OUTPUT_PATH}/{split}/masks/{filename}_{r}.png", mask_slice)

            #Augment only once per slice
            if split == "train":
                augmented = augment(image = img_slice, mask=mask_slice)

                aug_img = augmented["image"]
                aug_mask = augmented["mask"]
                cv2.imwrite(f"{OUTPUT_PATH}/{split}/images/{filename}_{r}_aug.png",aug_img)
                cv2.imwrite(f"{OUTPUT_PATH}/{split}/masks/{filename}_{r}_aug.png", aug_mask)
        

d:\Users\Asus\Liver_Tumor_Segmentation_Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# import sys
# !{sys.executable} -m pip install albumentations --no-cache-dir

   ---------------------------------------- 0.0/40.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.1 MB ? eta -:--:--
    --------------------------------------- 0.5/40.1 MB 764.3 kB/s eta 0:00:52
    --------------------------------------- 0.8/40.1 MB 1.0 MB/s eta 0:00:39
   - -------------------------------------- 1.3/40.1 MB 1.2 MB/s eta 0:00:33
   - -------------------------------------- 1.6/40.1 MB 1.3 MB/s eta 0:00:31
   -- ------------------------------------- 2.1/40.1 MB 1.5 MB/s eta 0:00:26
   -- ------------------------------------- 2.4/40.1 MB 1.5 MB/s eta 0:00:25
   -- ------------------------------------- 2.6/40.1 MB 1.5 MB/s eta 0:00:25
   -- ------------------------------------- 2.6/40.1 MB 1.5 MB/s eta 0:00:25
   -- ------------------------------------- 2.9/40.1 MB 1.3 MB/s eta 0:00:30
   --- ------------------------------------ 3.4/40.1 MB 1.4 MB/s eta 0:00:27
   --- -----------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
print("Train:", len(os.listdir("../processed_data/train/images")))
print("Val:", len(os.listdir("../processed_data/val/images")))
print("Test:", len(os.listdir("../processed_data/test/images")))

Train: 0
Val: 0
Test: 0


In [7]:
# Run preprocessing
from tqdm import tqdm
def process_split(patient_list, split):
    for patient in tqdm(patient_list, desc = f"Processing {split}"):
        process_patient(patient, split)

process_split(train_patients, "train")
process_split(val_patients,"val")
process_split(test_patients,"test")

Processing test: 100%|██████████| 16/16 [00:55<00:00,  3.47s/it]


In [8]:
print("Train images : ", len(os.listdir(f"{OUTPUT_PATH}/train/images")))
print("Val images : ", len(os.listdir(f"{OUTPUT_PATH}/val/images")))
print("Test images : ", len(os.listdir(f"{OUTPUT_PATH}/test/images")))

Train images :  22108
Val images :  2489
Test images :  2753


In [ ]:
import os
img_files = set(os.listdir("../processed_data/train/images"))
mask_files = set(os.listdir("../processed_data/train/masks"))
print("Mismatch:", len(img_files - mask_files))

Mismatch: 0


In [ ]:
import random
DATA_PATH = "../processed_data/train"
mask_files = os.listdir(f"{DATA_PATH}/masks")
sample = random.choice(mask_files)
mask = cv2.imread(f"{DATA_PATH}/masks/{sample}",0)
print("Mask value",np.unique(mask))

Mask value [0 1]


In [13]:
found_tumor = False

for file in mask_files[:200]:
    mask = cv2.imread(f"{DATA_PATH}/masks/{file}", 0)
    if 2 in np.unique(mask):
        print("Tumor class found ✅")
        found_tumor = True
        break

if not found_tumor:
    print("⚠️ No tumor class found")

Tumor class found ✅
